# NQ100 売買ルールバックテスト

Google Drive上のCSVを読み込み、東京時間の市場状態AIと売買シミュレーションを検証するためのColabノートブックです。

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
import sys
from pathlib import Path

# ColabでGitHubから開いた場合、必要に応じてリポジトリをcloneします。
REPO_DIR = Path('/content/NQ100')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/hon-daisuki/NQ100.git', str(REPO_DIR)],
        check=True,
    )

sys.path.insert(0, str(REPO_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import pandas as pd
import numpy as np

from src.labels import add_future_return_label, add_direction_label
from src.backtest import build_long_short_returns, summarize_returns
from src.metrics import describe_time_range, missing_summary

## データ読み込み

In [16]:
DATA_ROOT = Path('/content/drive')
CSV_CANDIDATES = [
    'USTEC_features_all.csv',
    'USTEC_1hour_features.csv',
]
PREFERRED_DIRS = [
    Path('/content/drive/MyDrive/CFD機械学習/backtest_ready'),
    Path('/content/drive/MyDrive/CFD機械学習'),
    Path('/content/drive/MyDrive'),
    DATA_ROOT,
]

def find_features_csv():
    checked = []
    for folder in PREFERRED_DIRS:
        for name in CSV_CANDIDATES:
            candidate = folder / name
            checked.append(str(candidate))
            if candidate.exists():
                return candidate

    matches = []
    for name in CSV_CANDIDATES:
        matches.extend(DATA_ROOT.rglob(name))
    if matches:
        return sorted(matches, key=lambda x: len(str(x)))[0]

    raise FileNotFoundError(
        '特徴量CSVが見つかりません。Driveに対象CSVがあるか、共有フォルダをMyDriveにショートカット追加してください。\n'
        + '探した候補:\n' + '\n'.join(checked)
    )

csv_path = find_features_csv()
print(f'Using CSV: {csv_path}')

df = pd.read_csv(csv_path)
df['日時'] = pd.to_datetime(df['日時'])
df = df.sort_values('日時').reset_index(drop=True)

describe_time_range(df)

Using CSV: /content/drive/MyDrive/CFD機械学習/NQ100/USTEC_1hour_features.csv


{'rows': 85831,
 'start': Timestamp('2011-10-01 00:00:00'),
 'end': Timestamp('2026-05-23 05:00:00'),
 'columns': ['日時',
  'open',
  'high',
  'low',
  'close',
  'return_1h',
  'range',
  'body',
  'upper_wick',
  'lower_wick',
  'hour',
  'dayofweek',
  'month',
  'sma_5',
  'sma_20',
  'ema_12',
  'ema_26',
  'macd',
  'macd_signal',
  'macd_hist',
  'bb_mid',
  'bb_std',
  'bb_upper',
  'bb_lower',
  'bb_width',
  'rsi_14',
  'atr_14']}

In [17]:
missing_summary(df)

,missing,missing_rate
bb_mid,19,0.000221
sma_20,19,0.000221
bb_lower,19,0.000221
bb_width,19,0.000221
bb_std,19,0.000221
bb_upper,19,0.000221
rsi_14,14,0.000163
atr_14,13,0.000151
sma_5,4,0.000047
return_1h,1,0.000012


## 東京時間フィルタとラベル作成

In [18]:
df['hour'] = df['日時'].dt.hour
df_tokyo = df[(df['hour'] >= 8) & (df['hour'] <= 12)].copy()

df_tokyo = add_future_return_label(df_tokyo, horizon=4, threshold=0.005)
df_tokyo = add_direction_label(df_tokyo, return_col='future_return_4h', threshold=0.005)

df_tokyo[['日時', 'close', 'future_return_4h', 'is_trend', 'signal']].tail()

,日時,close,future_return_4h,is_trend,signal
85809,2026-05-22 08:00:00,29549.9,0.000741,0,0
85810,2026-05-22 09:00:00,29564.5,NaN,0,0
85811,2026-05-22 10:00:00,29528.8,NaN,0,0
85812,2026-05-22 11:00:00,29548.1,NaN,0,0
85813,2026-05-22 12:00:00,29571.8,NaN,0,0


## 学習データ

In [19]:
features_trend = [
    'return_1h', 'range', 'body', 'upper_wick', 'lower_wick',
    'hour', 'dayofweek', 'month', 'sma_5', 'sma_20',
    'macd', 'macd_signal', 'macd_hist', 'bb_width', 'rsi_14', 'atr_14'
]

df_model = df_tokyo.dropna(subset=features_trend + ['is_trend', 'future_return_4h']).copy()
split_index = int(len(df_model) * 0.8)

X_train = df_model[features_trend].iloc[:split_index]
X_test = df_model[features_trend].iloc[split_index:]
y_train = df_model['is_trend'].iloc[:split_index]
y_test = df_model['is_trend'].iloc[split_index:]

len(X_train), len(X_test), y_train.mean(), y_test.mean()

(15000, 3751, np.float64(0.4706), np.float64(0.49853372434017595))

## LightGBM学習

In [20]:
!pip -q install lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_trend = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_trend.fit(X_train, y_train)
proba = model_trend.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('Accuracy =', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

[LightGBM] [Info] Number of positive: 7059, number of negative: 7941
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001337 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3339
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Accuracy = 0.6475606504932018
[[ 996  885]
 [ 437 1433]]
              precision    recall  f1-score   support

           0       0.70      0.53      0.60      1881
           1       0.62      0.77      0.68      1870

    accuracy                           0.65      3751
   macro avg       0.66      0.65      0.64      3751
weighted avg       0.66      0.65      0.64      3751



## Direction Model

Predict whether the future 4-hour move is down, flat, or up.

In [21]:
direction_threshold = 0.005

df_model['direction_signal'] = 0
df_model.loc[df_model['future_return_4h'] > direction_threshold, 'direction_signal'] = 1
df_model.loc[df_model['future_return_4h'] < -direction_threshold, 'direction_signal'] = -1

# Convert -1/0/1 signals to 0/1/2 classes for LightGBM multiclass training.
signal_to_class = {-1: 0, 0: 1, 1: 2}
class_to_signal = {0: -1, 1: 0, 2: 1}

y_dir = df_model['direction_signal'].map(signal_to_class)
y_dir_train = y_dir.iloc[:split_index]
y_dir_test = y_dir.iloc[split_index:]

df_model['direction_signal'].value_counts(normalize=True).sort_index()


,proportion
direction_signal,
-1,0.204096
0,0.523812
1,0.272092


In [22]:
model_dir = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_dir.fit(X_train, y_dir_train)
dir_proba = model_dir.predict_proba(X_test)
dir_class_pred = dir_proba.argmax(axis=1)
dir_signal_pred = pd.Series(dir_class_pred, index=X_test.index).map(class_to_signal)

print('Direction Accuracy =', accuracy_score(y_dir_test, dir_class_pred))
print(confusion_matrix(y_dir_test, dir_class_pred))
print(classification_report(
    y_dir_test,
    dir_class_pred,
    target_names=['down', 'flat', 'up'],
))


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001498 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3339
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 16
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Direction Accuracy = 0.4209544121567582
[[565 174  60]
 [867 935  79]
 [737 255  79]]
              precision    recall  f1-score   support

        down       0.26      0.71      0.38       799
        flat       0.69      0.50      0.58      1881
          up       0.36      0.07      0.12      1071

    accuracy                           0.42      3751
   macro avg       0.44      0.43      0.36      3751
weighted avg       0.50      0.42      0.41      3751



## Backtest With Predicted Direction

Use the trend model to select likely large-move periods, then use the direction model to choose long, short, or no trade. This cell does not use future direction as the trading signal.

In [23]:
bt = df_model.iloc[split_index:].copy()
bt['trend_proba'] = proba
bt['dir_confidence'] = dir_proba.max(axis=1)
bt['pred_direction_signal'] = dir_signal_pred.values

trend_threshold = 0.6
direction_confidence_threshold = 0.45

bt['trade_signal'] = np.where(
    (bt['trend_proba'] >= trend_threshold)
    & (bt['dir_confidence'] >= direction_confidence_threshold),
    bt['pred_direction_signal'],
    0,
)

returns = build_long_short_returns(
    bt,
    signal_col='trade_signal',
    return_col='future_return_4h',
    cost_per_trade=0.0002,
)

print('Signal counts:')
print(bt['trade_signal'].value_counts().sort_index())
print('\nBacktest summary:')
summarize_returns(returns)


Signal counts:
trade_signal
-1    1473
 0    2141
 1     137
Name: count, dtype: int64

Backtest summary:


{'trades': 2193.0,
 'total_return': -0.5553950165636456,
 'win_rate': 0.20181284990669154,
 'mean_return': -0.0001834731697055477,
 'max_drawdown': -0.6285588825160768}

In [24]:
datetime_col = df.columns[0]
bt[[datetime_col, 'close', 'future_return_4h', 'trend_proba', 'dir_confidence', 'pred_direction_signal', 'trade_signal']].tail(20)


,日時,close,future_return_4h,trend_proba,dir_confidence,pred_direction_signal,trade_signal
85718,2026-05-18 09:00:00,29033.1,0.003479,0.621940,0.764962,-1,-1
85719,2026-05-18 10:00:00,29021.3,0.002140,0.415762,0.524866,-1,0
85720,2026-05-18 11:00:00,29009.1,0.000138,0.383018,0.493697,0,0
85721,2026-05-18 12:00:00,29054.9,-0.002891,0.430647,0.705339,-1,0
85740,2026-05-19 08:00:00,29134.1,-0.004984,0.201289,0.755095,0,0
85741,2026-05-19 09:00:00,29083.4,-0.004576,0.507444,0.512986,-1,0
85742,2026-05-19 10:00:00,29013.1,-0.002620,0.465821,0.583561,-1,0
85743,2026-05-19 11:00:00,28970.9,0.000276,0.389142,0.562772,-1,0
85744,2026-05-19 12:00:00,28988.9,-0.003660,0.520373,0.477738,-1,0
85763,2026-05-20 08:00:00,28950.3,-0.001917,0.215634,0.837944,0,0
